In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1169, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 1169 (delta 137), reused 0 (delta 0), pack-reused 961 (from 2)
Receiving objects: 100% (1169/1169), 530.46 MiB | 40.71 MiB/s, done.
Resolving deltas: 100% (657/657), done.
Updating files: 100% (210/210), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'rp3beta_study.db'
json_name = 'rp3beta_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': True,
    'save_github': True,
    'candidates': '20',
    'n_trials': 1500
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'RP3Beta_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_rp3beta(optuna_trial):
    recommender_instance = RP3betaRecommender(URM_train)
    
    params = {
        'topK': optuna_trial.suggest_int("topK", 0, 1000),
        'alpha': optuna_trial.suggest_float("alpha", 0, 2.0),
        'beta': optuna_trial.suggest_float("beta", 0, 2.0),
        'normalize_similarity': optuna_trial.suggest_categorical("normalize_similarity", [True, False]),
        'implicit': True
    }
    
    recommender_instance.fit(**params)
    
    # Evaluate the model for Recall@50
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='RP3Beta_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_rp3beta, n_trials=config['n_trials'])

[I 2025-12-26 16:29:17,686] Using an existing study with name 'RP3Beta_Optimization' instead of creating a new one.


Previous trials found: 1000
RP3betaRecommender: Similarity column 6969 (100.0%), 1590.35 column/sec. Elapsed time 4.38 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 14.74 sec. Users per second: 1836


[I 2025-12-26 16:29:38,133] Trial 1000 finished with value: 0.24860228247661106 and parameters: {'topK': 53, 'alpha': 1.4072427680252257, 'beta': 0.4323516769613753, 'normalize_similarity': True}. Best is trial 252 with value: 0.25088063806278815.


RP3betaRecommender: Similarity column 6969 (100.0%), 1563.21 column/sec. Elapsed time 4.46 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 14.47 sec. Users per second: 1871


[I 2025-12-26 16:29:57,750] Trial 1001 finished with value: 0.2482247394596102 and parameters: {'topK': 73, 'alpha': 1.5152305031187088, 'beta': 0.2988405754256693, 'normalize_similarity': True}. Best is trial 252 with value: 0.25088063806278815.


...

RP3betaRecommender: Similarity column 6969 (100.0%), 1563.50 column/sec. Elapsed time 4.46 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 14.64 sec. Users per second: 1849


[I 2025-12-27 00:39:59,758] Trial 2498 finished with value: 0.24644918132810067 and parameters: {'topK': 89, 'alpha': 1.3886348139317324, 'beta': 0.3387251289015899, 'normalize_similarity': True}. Best is trial 252 with value: 0.25088063806278815.


RP3betaRecommender: Similarity column 6969 (100.0%), 866.12 column/sec. Elapsed time 8.05 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 17.90 sec. Users per second: 1512


[I 2025-12-27 00:40:28,366] Trial 2499 finished with value: 0.21699492090051048 and parameters: {'topK': 477, 'alpha': 1.2035525797009823, 'beta': 0.3860059509096982, 'normalize_similarity': True}. Best is trial 252 with value: 0.25088063806278815.


CPU times: user 7h 52min 47s, sys: 17min 57s, total: 8h 10min 44s
Wall time: 8h 11min 12s


In [17]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.250881 <= Saved: 0.250881
